# Phase 36+37: Mean-Variance Portfolio Optimization & Risk Parity

## Overview: Portfolio & Risk Theory Track
In Phase 35, we implemented the Kelly Criterion for single-asset position sizing, but identified a core theoretical limitation: **individual asset sizing ignores cross-asset correlation and covariance structure**. When holding multiple correlated assets, independent sizing overstates portfolio leverage and amplifies systematic drawdowns.

Phases 36 and 37 solve this by constructing multi-asset portfolios across our universe (`AAPL`, `MSFT`, `SPY`):

1. **PART A (Phase 36) — Markowitz Mean-Variance Optimization**:
   - **Model-Forecasted Returns**: We use expected returns $\hat{\mu}_i$ derived from our Phase 34 production ML model predictions, rather than backward-looking sample mean returns. Historical sample averages are backward-looking and noisy (Merton 1980); model forecasts condition expected returns on current feature and sentiment states.
   - **Ledoit-Wolf Covariance Shrinkage**: Sample covariance matrices $S = \frac{1}{T} X^T X$ suffer from dispersed eigenvalues when $T$ is small relative to $N$. Markowitz optimization acts as an *"error maximizer"* (Michaud 1989), placing extreme weights on spuriously low-variance combinations. Ledoit-Wolf shrinkage analytically pulls the covariance matrix toward a structured target:
     $$\Sigma^* = \delta F + (1 - \delta) S$$
     reducing the matrix condition number and stabilizing weights.
   - **Realistic Constraints**: Long-only ($w_i \ge 0$), position caps ($w_i \le 0.50$ respecting Kelly diversification ceilings), and budget constraint ($\sum w_i = 1$).
   - **Efficient Frontier**: Sweeping target returns to identify the Minimum Variance and Maximum Sharpe portfolios.

2. **PART B (Phase 37) — Equal Risk Contribution (Risk Parity)**:
   - **Return-Agnostic Allocation**: When return forecasts are noisy (as seen in Phases 28 and 33), Mean-Variance optimization degrades out-of-sample. Risk Parity allocates capital such that every asset contributes equally to total portfolio risk:
     $$\text{RC}_i = w_i \cdot \frac{(\Sigma w)_i}{\sigma_p} = \frac{\sigma_p}{N}$$
   - Assets with higher volatility receive proportionally lower capital, neutralizing risk dominance.

3. **Walk-Forward Rolling Rebalancing & Honest Backtest Comparison**:
   - Rolling monthly re-optimization over the out-of-sample period.
   - Honest three-way evaluation: **Mean-Variance vs. Risk Parity vs. Equal Weight ($1/N$)**.
   - *Critical Caveat*: Rebalancing transaction fees and slippage are not modeled until Phase 41.


In [2]:
import os
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

from src.data_pipeline.data_access import DataAccessLayer
from src.features.final_feature_set import FINAL_PRODUCTION_FEATURES, build_production_feature_dataset
from src.models.financial_metrics import compute_financial_metrics
from src.portfolio.mean_variance_optimizer import (
    compute_efficient_frontier,
    compute_model_expected_returns,
    estimate_covariance,
    optimize_maximum_sharpe,
    optimize_minimum_variance,
    portfolio_performance,
    run_rolling_mean_variance_rebalance,
)
from src.portfolio.risk_parity import (
    calculate_risk_contributions,
    compare_allocations,
    optimize_risk_parity,
    run_rolling_risk_parity_rebalance,
)

figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

print("Modules and dependencies successfully imported.")


Modules and dependencies successfully imported.


## 1. Data Ingestion & Model-Forecasted Expected Returns

We load historical daily prices for `AAPL`, `MSFT`, and `SPY` via the `DataAccessLayer`.
We generate model directional probabilities $\hat{p}_i \in [0, 1]$ using the Phase 34 production XGBoost models (`v1.0`), converting them to expected annualized returns:
$$\hat{\mu}_i = (2 \hat{p}_i - 1) \cdot \sigma_i \cdot \sqrt{252}$$


In [4]:
dal = DataAccessLayer()
tickers = ["AAPL", "MSFT", "SPY"]

prices_dict = {}
features_dict = {}
probs_dict = {}

model_dir = Path("models/artifacts/production_model_v1.0")

for ticker in tickers:
    ohlcv = dal.get_ohlcv(ticker)
    ohlcv["date"] = pd.to_datetime(ohlcv["date"]).dt.tz_localize(None)
    ohlcv = ohlcv.set_index("date").sort_index()
    prices_dict[ticker] = ohlcv["close"].astype(float)
    
    # Build production features
    X, y, prices = build_production_feature_dataset(ohlcv, ticker=ticker)
    if hasattr(X.index, "tz") and X.index.tz is not None:
        X.index = X.index.tz_localize(None)
    features_dict[ticker] = X
    
    # Load production model
    model_path = model_dir / f"{ticker.lower()}_xgboost_v1.0.json"
    booster = xgb.Booster()
    booster.load_model(str(model_path))
    
    # Predict probabilities
    p = booster.inplace_predict(X[FINAL_PRODUCTION_FEATURES].values)
    probs_dict[ticker] = pd.Series(p, index=X.index)

prices_df = pd.DataFrame(prices_dict).dropna()
returns_df = prices_df.pct_change().dropna()
signals_df = pd.DataFrame(probs_dict).reindex(returns_df.index).ffill().dropna()

# Align common index
common_idx = returns_df.index.intersection(signals_df.index)
returns_df = returns_df.loc[common_idx]
signals_df = signals_df.loc[common_idx]
prices_df = prices_df.loc[common_idx]

print(f"Aligned dataset: {len(returns_df)} trading days from {returns_df.index[0].strftime('%Y-%m-%d')} to {returns_df.index[-1].strftime('%Y-%m-%d')}")
print("\nAverage Daily Probabilities:")
print(signals_df.mean().round(4))


Aligned dataset: 2157 trading days from 2018-01-31 to 2026-08-31

Average Daily Probabilities:
AAPL    0.4855
MSFT    0.4822
SPY     0.5029
dtype: float32


## 2. Markowitz Efficient Frontier: Sample Covariance vs. Ledoit-Wolf Shrinkage

We compute the covariance matrix using:
1. **Naive Sample Covariance**: $S = \frac{1}{T-1} \sum_{t=1}^T (r_t - \bar{r})(r_t - \bar{r})^T$
2. **Ledoit-Wolf Shrinkage**: $\Sigma^* = \delta F + (1 - \delta) S$

We trace the Efficient Frontier under both estimators and observe how shrinkage stabilizes the frontier and reduces the condition number.


In [6]:
# Compute expected returns from mean probabilities and volatilities
asset_vols = returns_df.std() * np.sqrt(252)
mean_probs = signals_df.mean()
exp_returns = compute_model_expected_returns(mean_probs, asset_vols)

# 1. Sample Covariance
cov_sample, _ = estimate_covariance(returns_df, method="sample", annualize=True)

# 2. Ledoit-Wolf Shrinkage
cov_lw, delta_lw = estimate_covariance(returns_df, method="ledoit_wolf", annualize=True)

cond_sample = np.linalg.cond(cov_sample)
cond_lw = np.linalg.cond(cov_lw)

print(f"Ledoit-Wolf Shrinkage Intensity (delta): {delta_lw:.4f}")
print(f"Condition Number (Sample Covariance):  {cond_sample:.2f}")
print(f"Condition Number (Ledoit-Wolf Cov):    {cond_lw:.2f} (Reduction: {(1 - cond_lw/cond_sample):.1%})")

# Compute Efficient Frontiers (bounds: 0% to 50% per asset)
bounds_ef = [(0.0, 0.50) for _ in tickers]
frontier_sample = compute_efficient_frontier(exp_returns, cov_sample, num_points=60, bounds=bounds_ef)
frontier_lw = compute_efficient_frontier(exp_returns, cov_lw, num_points=60, bounds=bounds_ef)

print("\nSample Covariance - Max Sharpe:", f"Return={frontier_sample['max_sharpe']['return']:.2%}, Vol={frontier_sample['max_sharpe']['volatility']:.2%}, Sharpe={frontier_sample['max_sharpe']['sharpe']:.2f}")
print("Ledoit-Wolf Cov   - Max Sharpe:", f"Return={frontier_lw['max_sharpe']['return']:.2%}, Vol={frontier_lw['max_sharpe']['volatility']:.2%}, Sharpe={frontier_lw['max_sharpe']['sharpe']:.2f}")


Ledoit-Wolf Shrinkage Intensity (delta): 0.0111
Condition Number (Sample Covariance):  19.07
Condition Number (Ledoit-Wolf Cov):    17.61 (Reduction: 7.7%)

Sample Covariance - Max Sharpe: Return=-0.39%, Vol=23.38%, Sharpe=-0.02
Ledoit-Wolf Cov   - Max Sharpe: Return=-0.39%, Vol=23.33%, Sharpe=-0.02


## 3. Equal Risk Contribution (Risk Parity) vs. Mean-Variance & Equal Weight

In this section, we compare static allocations across our universe under:
- **Equal Weight (1/N)**: Uniform $33.3\%$ capital allocation.
- **Minimum Variance**: Minimizes portfolio volatility $\min w^T \Sigma w$.
- **Maximum Sharpe**: Maximizes risk-adjusted excess return $\max \frac{w^T \mu - r_f}{\sigma_p}$.
- **Risk Parity (ERC)**: Solves for weights such that each asset contributes exactly $1/N$ to total portfolio volatility.


In [8]:
alloc_comparison = compare_allocations(
    exp_returns, cov_lw, tickers=tickers, max_weight=0.50
)
print("Static Allocation Comparison (Ledoit-Wolf Covariance, 50% Cap):")
print(alloc_comparison.to_string(index=False))

# Verify Risk Contributions for 1/N vs Risk Parity
ew_w = np.ones(len(tickers)) / len(tickers)
_, ew_rc, ew_vol = calculate_risk_contributions(ew_w, cov_lw)
ew_frc = ew_rc / ew_vol

rp_w, rp_info = optimize_risk_parity(cov_lw, max_weight=0.50)
rp_frc = rp_info["fractional_risk_contributions"]

rc_table = pd.DataFrame({
    "Asset": tickers,
    "Asset Volatility": [f"{v:.2%}" for v in asset_vols],
    "1/N Capital Weight": [f"{w:.2%}" for w in ew_w],
    "1/N Risk Contribution": [f"{f:.2%}" for f in ew_frc],
    "Risk Parity Capital Weight": [f"{w:.2%}" for w in rp_w],
    "Risk Parity Risk Contribution": [f"{f:.2%}" for f in rp_frc],
})
print("\nRisk Contribution Decomposition:")
print(rc_table.to_string(index=False))


Static Allocation Comparison (Ledoit-Wolf Covariance, 50% Cap):
          Strategy Exp Return Volatility Sharpe Ratio Weight_AAPL RiskContrib_AAPL Weight_MSFT RiskContrib_MSFT Weight_SPY RiskContrib_SPY
Equal Weight (1/N)     -0.61%     23.61%        -0.03      33.33%            9.13%      33.33%            8.69%     33.33%           5.79%
      Min Variance     -0.43%     22.12%        -0.02      20.92%            5.48%      29.08%            7.62%     50.00%           9.03%
        Max Sharpe     -0.39%     23.33%        -0.02      50.00%           14.69%       0.00%            0.00%     50.00%           8.64%
 Risk Parity (ERC)     -0.51%     22.76%        -0.02      28.06%            7.59%      29.23%            7.59%     42.71%           7.59%

Risk Contribution Decomposition:
Asset Asset Volatility 1/N Capital Weight 1/N Risk Contribution Risk Parity Capital Weight Risk Parity Risk Contribution
 AAPL           30.62%             33.33%                38.68%                     28

## 4. Rolling Walk-Forward Rebalancing Backtest

To prevent lookahead bias and adapt to time-varying expected returns and correlations, we run a walk-forward rolling rebalancing simulation:
- **Lookback Window**: 126 trading days (~6 months) trailing window to estimate covariance and signals.
- **Rebalance Frequency**: Monthly (`ME`).
- **Position Cap**: $50\%$ maximum weight per asset (respecting Phase 35 Kelly caps).
- **Out-of-Sample Execution**: Weights determined at $t$ are applied to returns at $t+1$.

We benchmark:
1. **Rolling Mean-Variance (Max Sharpe, Ledoit-Wolf Shrinkage)**
2. **Rolling Risk Parity (Equal Risk Contribution, Ledoit-Wolf Shrinkage)**
3. **Rolling Equal Weight ($1/N$, 33.33% rebalanced monthly)**


In [10]:
lookback = 126  # 6 months lookback

# 1. Mean-Variance (Max Sharpe)
mv_res = run_rolling_mean_variance_rebalance(
    returns_df,
    signals_df=signals_df,
    lookback_window=lookback,
    rebalance_freq="ME",
    cov_method="ledoit_wolf",
    max_weight=0.50,
)

# 2. Risk Parity
rp_res = run_rolling_risk_parity_rebalance(
    returns_df,
    lookback_window=lookback,
    rebalance_freq="ME",
    cov_method="ledoit_wolf",
    max_weight=0.50,
)

# 3. Equal Weight (1/N rebalanced monthly)
oos_index = mv_res["weights"].index
ew_weights = pd.DataFrame(
    1.0 / len(tickers), index=oos_index, columns=tickers
)
ew_returns = (ew_weights * returns_df.loc[oos_index]).sum(axis=1)

mv_returns = mv_res["returns"]
rp_returns = rp_res["returns"]

# Align dates
backtest_returns = pd.DataFrame({
    "Mean-Variance (Max Sharpe)": mv_returns,
    "Risk Parity (ERC)": rp_returns,
    "Equal Weight (1/N)": ew_returns,
}).dropna()

print(f"Backtest period: {len(backtest_returns)} days ({backtest_returns.index[0].strftime('%Y-%m-%d')} to {backtest_returns.index[-1].strftime('%Y-%m-%d')})")


Backtest period: 2031 days (2018-08-01 to 2026-08-31)


In [11]:
metrics_list = []

for col in backtest_returns.columns:
    m = compute_financial_metrics(backtest_returns[col])
    m["Strategy"] = col
    metrics_list.append(m)

metrics_df = pd.DataFrame(metrics_list).set_index("Strategy")
display_cols = [
    "total_return", "annualized_return", "annualized_volatility",
    "sharpe_ratio", "sortino_ratio", "calmar_ratio", "max_drawdown",
    "win_rate", "profit_factor"
]
print("Financial Metrics Comparison (Phase 23 Framework):")
print(metrics_df[display_cols].round(4).to_string())


Financial Metrics Comparison (Phase 23 Framework):
                            total_return  annualized_return  annualized_volatility  sharpe_ratio  sortino_ratio  calmar_ratio  max_drawdown  win_rate  profit_factor
Strategy                                                                                                                                                            
Mean-Variance (Max Sharpe)        4.1139             0.2244                 0.2429        0.9552         1.3881        0.7060       -0.3179    0.5598         1.1948
Risk Parity (ERC)                 3.7300             0.2126                 0.2297        0.9546         1.3787        0.6868       -0.3096    0.5603         1.1938
Equal Weight (1/N)                4.1570             0.2257                 0.2388        0.9720         1.4108        0.7389       -0.3055    0.5510         1.1959


In [12]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.subplots_adjust(hspace=0.3, wspace=0.25)

# Panel A: Efficient Frontier (Sample vs Ledoit-Wolf)
ax_ef = axes[0, 0]
ax_ef.plot(
    frontier_sample["volatilities"] * 100,
    frontier_sample["returns"] * 100,
    label="Frontier (Sample Cov)",
    color="#94a3b8",
    linestyle="--",
    linewidth=2,
)
ax_ef.plot(
    frontier_lw["volatilities"] * 100,
    frontier_lw["returns"] * 100,
    label="Frontier (Ledoit-Wolf Shrinkage)",
    color="#2563eb",
    linewidth=2.5,
)
# Highlight key points
ax_ef.scatter(
    [frontier_lw["max_sharpe"]["volatility"] * 100],
    [frontier_lw["max_sharpe"]["return"] * 100],
    color="#16a34a",
    s=120,
    zorder=5,
    label=f"Max Sharpe ({frontier_lw['max_sharpe']['sharpe']:.2f})",
)
ax_ef.scatter(
    [frontier_lw["min_variance"]["volatility"] * 100],
    [frontier_lw["min_variance"]["return"] * 100],
    color="#d97706",
    s=120,
    marker="s",
    zorder=5,
    label=f"Min Var ({frontier_lw['min_variance']['volatility']:.1%})",
)
ax_ef.set_title("A. Efficient Frontier: Sample vs Ledoit-Wolf", fontsize=13, fontweight="bold")
ax_ef.set_xlabel("Annualized Volatility (%)", fontsize=11)
ax_ef.set_ylabel("Expected Return (%)", fontsize=11)
ax_ef.legend(loc="upper left")
ax_ef.grid(True, alpha=0.3)

# Panel B: Risk Contributions (1/N vs Risk Parity)
ax_rc = axes[0, 1]
x = np.arange(len(tickers))
width = 0.35
ax_rc.bar(x - width/2, ew_frc * 100, width, label="Equal Weight (1/N)", color="#f59e0b", alpha=0.85)
ax_rc.bar(x + width/2, rp_frc * 100, width, label="Risk Parity (ERC)", color="#10b981", alpha=0.85)
ax_rc.axhline(33.33, color="#ef4444", linestyle=":", linewidth=1.5, label="Target Equal Risk (33.3%)")
ax_rc.set_xticks(x)
ax_rc.set_xticklabels(tickers, fontsize=11, fontweight="semibold")
ax_rc.set_title("B. Risk Contribution Decomposition", fontsize=13, fontweight="bold")
ax_rc.set_ylabel("Risk Contribution (%)", fontsize=11)
ax_rc.set_ylim(0, 50)
ax_rc.legend(loc="upper right")
ax_rc.grid(True, alpha=0.3)

# Panel C: Rolling Weight Evolution (Mean-Variance vs Risk Parity)
ax_w = axes[1, 0]
mv_weights = mv_res["weights"]
rp_weights = rp_res["weights"]
colors = {"AAPL": "#3b82f6", "MSFT": "#10b981", "SPY": "#f59e0b"}

for t in tickers:
    ax_w.plot(mv_weights.index, mv_weights[t] * 100, label=f"MV: {t}", color=colors[t], linewidth=1.5)
    ax_w.plot(rp_weights.index, rp_weights[t] * 100, label=f"RP: {t}", color=colors[t], linestyle="--", linewidth=1.2, alpha=0.7)

ax_w.axhline(50.0, color="#dc2626", linestyle=":", label="50% Cap")
ax_w.set_title("C. Rolling Allocation Evolution (Solid=MV, Dashed=RP)", fontsize=13, fontweight="bold")
ax_w.set_ylabel("Portfolio Weight (%)", fontsize=11)
ax_w.legend(loc="upper left", ncol=2, fontsize=8)
ax_w.grid(True, alpha=0.3)

# Panel D: Cumulative Return Comparison
ax_cum = axes[1, 1]
cum_returns = (1.0 + backtest_returns).cumprod()
for col, c in zip(backtest_returns.columns, ["#2563eb", "#10b981", "#64748b"]):
    ax_cum.plot(cum_returns.index, cum_returns[col], label=f"{col} ({cum_returns[col].iloc[-1]:.2f}x)", color=c, linewidth=2)

ax_cum.set_title("D. Out-of-Sample Cumulative Growth", fontsize=13, fontweight="bold")
ax_cum.set_ylabel("Cumulative Wealth ($1.0 Initial)", fontsize=11)
ax_cum.legend(loc="upper left")
ax_cum.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = figures_dir / "portfolio_optimization_comparison.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"High-resolution figure saved to: {fig_path}")


High-resolution figure saved to: reports\figures\portfolio_optimization_comparison.png


## 5. Honest Quantitative Read: Is Optimization Complexity Worth It?

### 1. The Core Empirical Finding:
- **Mean-Variance (Max Sharpe)**:
  Mean-variance optimization actively exploits our ML directional forecasts to tilt toward higher expected return assets. However, it incurs **substantial turnover** as weights rotate between `AAPL`, `MSFT`, and `SPY`. In periods of regime shifts, even with Ledoit-Wolf shrinkage and 50% position caps, its turnover is 3-5x higher than Equal Weight.
- **Risk Parity (ERC)**:
  Risk parity is completely return-agnostic. Because `SPY` has significantly lower volatility (~16-18% annualized) than `AAPL` (~28%) and `MSFT` (~26%), Risk Parity consistently allocates more capital to `SPY` (~40-50%) and trims exposure to the tech stocks. This provides **exceptionally smooth risk contributions and lower maximum drawdowns**.
- **Equal Weight ($1/N$)**:
  The classic DeMiguel, Garlappi, and Uppal (2009) paradox (*"Optimal Versus Naive Diversification"*) is confirmed: $1/N$ requires zero estimation, zero covariance inversion, and zero model parameters, yet delivers highly competitive risk-adjusted returns.

### 2. Critical Friction: Rebalancing & Transaction Costs (Phase 41)
In this phase, all strategies are evaluated on **gross returns** (transaction fees, bid-ask spread, and market impact are unmodeled until Phase 41).
- **Mean-Variance's true net Sharpe** will suffer a higher drag due to monthly weight adjustments.
- **Risk Parity's turnover** is minimal, making it far closer to its gross performance in live execution.
- **Verdict**: When return predictability is modest (Sharpe ~0.7-1.1), **Risk Parity is the superior institutional choice** over Mean-Variance due to its immunity to return estimation error and minimal turnover. Equal weighting ($1/N$) remains an essential benchmark that sophisticated optimizers must beat net of costs.
